# Bronze Layer - Olist E-commerce Dataset

**Objetivo:** Ingestão de dados brutos (raw data) na camada Bronze do Data Lakehouse seguindo a arquitetura Medalhão.

**Fontes de Dados:**
- Arquivos CSV do dataset Olist (e-commerce brasileiro)
- API do Banco Central do Brasil (cotação do dólar)

**Regras de Negócio:**
- Preservação dos dados originais sem transformações
- Adição de timestamp de ingestão para rastreabilidade
- Uso de Delta Lake para garantir ACID compliance

## 1. Configuração da Infraestrutura

Criação da estrutura de dados no Unity Catalog seguindo a arquitetura Medalhão (Bronze, Silver, Gold).

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS medalhao;
CREATE SCHEMA IF NOT EXISTS medalhao.bronze;
CREATE SCHEMA IF NOT EXISTS medalhao.silver;
CREATE SCHEMA IF NOT EXISTS medalhao.gold;
CREATE SCHEMA IF NOT EXISTS medalhao.landing;

CREATE VOLUME IF NOT EXISTS medalhao.landing.olist_dataset;

## 2. Configuração de Mapeamento de Tabelas

Definição da convenção de nomenclatura e localização dos arquivos fonte.

In [0]:
# padronizando nomes de arquivos
mapeamento_tabelas = {
    "olist_customers_dataset.csv": "tb_customers",
    "olist_geolocation_dataset.csv": "tb_geolocalizacao",
    "olist_order_items_dataset.csv": "tb_order_items",
    "olist_order_payments_dataset.csv": "tb_order_payments",
    "olist_order_reviews_dataset.csv": "tb_order_reviews",
    "olist_orders_dataset.csv": "tb_orders",
    "olist_products_dataset.csv": "tb_products",
    "olist_sellers_dataset.csv": "tb_sellers",
    "product_category_name_translation.csv": "tb_product_category_name_translation"
}

landing_path = "/Volumes/medalhao/landing/olist_dataset/"

## 3. Ingestão de Arquivos CSV do Dataset Olist

Processamento e carga dos arquivos CSV na camada Bronze com preservação dos dados originais.

In [0]:
# lendo arquivos e gravando na Bronze
from pyspark.sql import functions as F

for arquivo, tabela in mapeamento_tabelas.items():
    print(f"Processando: {arquivo} - bronze.{tabela}")

    df = (spark.read
          .option("header", "true")
          .option("inferSchema", "true")
          .option("multiLine", "true") 
          .option("escape", '"')    
          .csv(f"{landing_path}{arquivo}"))
    
    # adicionando timestamp_ingestion
    df_bronze = df.withColumn("timestamp_ingestion", F.current_timestamp())

    # visualizando os dados
    print(f"Schema da tabela {tabela}:")
    df_bronze.printSchema()

    print(f"Prévia dos dados da tabela {tabela}:")
    display(df_bronze.limit(5))
    
    #gravando na bronze
    (df_bronze.write
     .format("delta")
     .mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable(f"medalhao.bronze.{tabela}"))

Processando: olist_customers_dataset.csv - bronze.tb_customers
Schema da tabela tb_customers:
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_gender: string (nullable = true)
 |-- customer_birth_date: date (nullable = true)
 |-- customer_age: integer (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = false)

Prévia dos dados da tabela tb_customers:


customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_name,customer_gender,customer_birth_date,customer_age,timestamp_ingestion
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,Zoe Nogueira,F,1956-09-07,70,2026-04-04T17:48:18.591Z
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,Liam Viana,M,1974-09-23,52,2026-04-04T17:48:18.591Z
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,Diego Aparecida,M,1964-05-20,62,2026-04-04T17:48:18.591Z
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,Marcos Vinicius Azevedo,M,1967-01-14,59,2026-04-04T17:48:18.591Z
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,Srta. Juliana Siqueira,F,1950-08-01,76,2026-04-04T17:48:18.591Z


Processando: olist_geolocation_dataset.csv - bronze.tb_geolocalizacao
Schema da tabela tb_geolocalizacao:
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = false)

Prévia dos dados da tabela tb_geolocalizacao:


geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,timestamp_ingestion
1037,-23.54562128115268,-46.63929204800168,sao paulo,SP,2026-04-04T17:48:24.116Z
1046,-23.546081127035535,-46.64482029837157,sao paulo,SP,2026-04-04T17:48:24.116Z
1046,-23.54612896641469,-46.64295148361138,sao paulo,SP,2026-04-04T17:48:24.116Z
1041,-23.5443921648681,-46.63949930627844,sao paulo,SP,2026-04-04T17:48:24.116Z
1035,-23.541577961711493,-46.64160722329613,sao paulo,SP,2026-04-04T17:48:24.116Z


Processando: olist_order_items_dataset.csv - bronze.tb_order_items
Schema da tabela tb_order_items:
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = false)

Prévia dos dados da tabela tb_order_items:


...

product_id,product_name,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,timestamp_ingestion
1e9e8ef04dbcff4541ed26657ea517e5,Perfume Premium,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,2026-04-04T17:48:44.201Z
3aa071139cb16b67ca9e5dea641aaa2f,Conjunto de Pincéis,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,2026-04-04T17:48:44.201Z
96bd76ec8810374ed1b65e291975717f,Barraca de Camping,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,2026-04-04T17:48:44.201Z
cef67bcfe19066a932b7673e239eb23d,Chupeta Premium,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,2026-04-04T17:48:44.201Z
9dc1a7de274444849c219cff195d0b71,Vassoura Mágica,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,2026-04-04T17:48:44.201Z


Processando: olist_sellers_dataset.csv - bronze.tb_sellers
Schema da tabela tb_sellers:
root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)
 |-- seller_name: string (nullable = true)
 |-- seller_registration_date: date (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = false)

Prévia dos dados da tabela tb_sellers:


seller_id,seller_zip_code_prefix,seller_city,seller_state,seller_name,seller_registration_date,timestamp_ingestion
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,Dr. Pedro Miguel Vasconcelos,2018-06-25,2026-04-04T17:48:47.495Z
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,Gabriela da Cruz,2018-09-21,2026-04-04T17:48:47.495Z
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,Luigi Viana,2018-04-03,2026-04-04T17:48:47.495Z
c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP,Beatriz Brito,2018-12-20,2026-04-04T17:48:47.495Z
51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,Gabrielly Pastor,2017-08-24,2026-04-04T17:48:47.495Z


Processando: product_category_name_translation.csv - bronze.tb_product_category_name_translation
Schema da tabela tb_product_category_name_translation:
root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)
 |-- timestamp_ingestion: timestamp (nullable = false)

Prévia dos dados da tabela tb_product_category_name_translation:


product_category_name,product_category_name_english,timestamp_ingestion
beleza_saude,health_beauty,2026-04-04T17:48:50.385Z
informatica_acessorios,computers_accessories,2026-04-04T17:48:50.385Z
automotivo,auto,2026-04-04T17:48:50.385Z
cama_mesa_banho,bed_bath_table,2026-04-04T17:48:50.385Z
moveis_decoracao,furniture_decor,2026-04-04T17:48:50.385Z


## 4. Ingestão de Dados de Cotação do Dólar

Coleta de dados de cotação do dólar via API do Banco Central do Brasil para análises de preço em moeda estrangeira.

In [0]:
import requests
import pandas as pd
from pyspark.sql import functions as F

# criando os parâmetros com o formato certo
dbutils.widgets.text("data_inicio", "09-01-2016") 
dbutils.widgets.text("data_fim", "10-31-2018")

# resgatando os valores digitados nos parâmetros
data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

print(f"Buscando cotação do dólar de {data_inicio} até {data_fim}...")

url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

resposta = requests.get(url)

if resposta.status_code == 200:
    dados_json = resposta.json()['value']
    
    # convertendo o JSON pra df pandas e depois pra spark
    pdf_dolar = pd.DataFrame(dados_json)
    df_dolar = spark.createDataFrame(pdf_dolar)
    
    df_bronze_dolar = df_dolar.withColumn("timestamp_ingestion", F.current_timestamp())
    
    # 7. Salvando a tabela na Bronze
    (df_bronze_dolar.write
     .format("delta")
     .mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable("medalhao.bronze.tb_cotacao_dolar"))
     
    print("Tabela bronze.tb_cotacao_dolar criada.")
else:
    print(f"Erro ao acessar a API: status code {resposta.status_code}")

Buscando cotação do dólar de 09-01-2016 até 10-31-2018...
Tabela bronze.tb_cotacao_dolar criada.
